[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhiyunli/cpsc5610-labs/blob/main/week4_lab.ipynb)

# Week 4 Lab: Building Neural Networks with PyTorch

This lab mirrors the Week 4 slides. Each section embeds the relevant slide, gives
you a short prompt, and hands you a partially-filled code cell. Replace every
`[YOUR CODE]` with your own implementation. The scaffolding around it
(imports, variable names, structure) is fixed; you focus on the critical lines.

**How to use:** run cells top to bottom. Do NOT skip ahead until the previous
block runs. Later sections reuse names (`X_train`, `model`, `device`, etc.) from
earlier ones.


![slide-2](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-02.jpg)

**Chapter roadmap.** Five parts: Fundamentals, Regression, Custom Modules,
Image Classification, Production Readiness. We will do all five below.

## Setup

Run this setup cell once. If you're missing a package, install it now
(`pip install torch torchvision torchmetrics optuna`).

In [ ]:
import sys
assert sys.version_info >= (3, 10), "Python 3.10+ required"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

print("torch version:", torch.__version__)


---
# Part 1 — PyTorch Fundamentals
Tensors, math ops, NumPy interop, device selection, autograd, training loops.

![slide-5](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-05.jpg)

## 1.1 The Tensor

A tensor is a NumPy-like multidimensional array with two extras: it can live on
a GPU, and it tracks operations for automatic differentiation.

**Your task:** create a 2x3 float tensor `X` with the values
`[[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]]`, then print its `shape`, `dtype`, and the
element at row 0, column 1.

In [ ]:
# Build the tensor
X = [YOUR CODE]  # shape (2, 3), float32

# Inspect it
print("shape:", [YOUR CODE])   # hint: .shape
print("dtype:", [YOUR CODE])   # hint: .dtype
print("X[0, 1] =", [YOUR CODE])
X


![slide-8](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-08.jpg)

## 1.2 Common Math Operations

Elementwise ops, reductions, and matmul all behave like NumPy.

**Your task:** compute, for the `X` above:
1. elementwise `exp`
2. overall `mean`
3. `X @ X.T` (matrix product)

In [ ]:
exp_X = [YOUR CODE]     # elementwise exp
mean_X = [YOUR CODE]    # scalar mean
gram = [YOUR CODE]      # X @ X.T, shape (2, 2)

print(exp_X)
print("mean:", mean_X)
print("gram:\n", gram)


![slide-9](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-09.jpg)

## 1.3 NumPy Interoperability

Tensors and ndarrays share memory when you use `from_numpy` / `.numpy()`.

**Your task:** round-trip an ndarray -> tensor -> ndarray and confirm equality.

In [ ]:
arr = np.array([[1., 4., 7.], [2., 3., 6.]], dtype=np.float32)

# tensor that views the same bytes as `arr`
T_from_np = [YOUR CODE]    # hint: torch.from_numpy

# ndarray that views the same bytes as the tensor
arr_back = [YOUR CODE]     # hint: .numpy()

print(T_from_np)
print(np.array_equal(arr, arr_back))


![slide-10](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-10.jpg)

## 1.4 Float Precision

PyTorch defaults to `float32`, not `float64` like NumPy. Keep this in mind when
you convert from numpy — cast explicitly.

**Your task:** make a float64 array, convert to a tensor, then build a second
tensor that is explicitly `float32`. Compare their memory footprints.

In [ ]:
a64 = np.arange(1_000_000, dtype=np.float64)

T64 = [YOUR CODE]   # from a64 without changing dtype
T32 = [YOUR CODE]   # from a64 but cast to float32

print("T64 dtype:", T64.dtype, "bytes:", T64.element_size() * T64.numel())
print("T32 dtype:", T32.dtype, "bytes:", T32.element_size() * T32.numel())


![slide-11](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-11.jpg)

## 1.5 In-Place Operations

Methods ending in `_` modify the tensor in place and return it. They can be
memory-efficient but are **incompatible with autograd on leaf tensors** — don't
use them on `requires_grad=True` tensors.

**Your task:** set all negative entries of `Y` to zero using an in-place ReLU,
then verify the original storage was modified.

In [ ]:
Y = torch.tensor([[-1.0, 2.0, -3.0], [4.0, -5.0, 6.0]])
Y_before_id = id(Y)

[YOUR CODE]   # apply ReLU in place on Y  (hint: method name ends with _)

print(Y)
print("same object?", id(Y) == Y_before_id)


![slide-13](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-13.jpg)

## 1.6 Device Selection

The standard pattern checks for CUDA, then Apple MPS, falling back to CPU.
Assign the device string to `device` — every subsequent cell will reuse it.

In [ ]:
if [YOUR CODE]:          # CUDA available?
    device = "cuda"
elif [YOUR CODE]:        # MPS available? (Apple Silicon)
    device = "mps"
else:
    device = "cpu"

print("device:", device)


![slide-18](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-18.jpg)

## 1.7 Automatic Differentiation

`requires_grad=True` turns a tensor into a leaf in the autograd graph.
`f.backward()` walks that graph backward and populates `.grad` on every
leaf that contributed.

**Your task:** compute the gradient of `f(x) = x^2` at `x = 5` — the answer
should be `10.0`.

In [ ]:
x = [YOUR CODE]       # scalar tensor 5.0 with gradient tracking
f = [YOUR CODE]       # f = x ** 2

[YOUR CODE]           # trigger backward pass

print("f =", f.item(), "  df/dx =", x.grad.item())


![slide-20](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-20.jpg)

## 1.8 How Autograd Tracks the Graph

Every non-leaf tensor has a `.grad_fn` pointing back to the op that produced
it. Inspect it below — no code to fill in, just read the output.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
c = a * b + a.sin()

print("c =", c.item())
print("c.grad_fn =", c.grad_fn)
print("c.grad_fn.next_functions =", c.grad_fn.next_functions)


![slide-23](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-23.jpg)

## 1.9 Gradients Accumulate

Silent bug: `.grad` is **added to** on every `backward()` call. You must zero it
between steps, either with `x.grad.zero_()` or `optimizer.zero_grad()`.

**Your task:** call `f.backward()` twice without zeroing, then confirm the stored
gradient is `2 * (2x)` = `20.0` instead of the expected `10.0`.

In [ ]:
x = torch.tensor(5.0, requires_grad=True)
f = x ** 2
f.backward()
print("after 1st backward:", x.grad.item())

f = x ** 2                # fresh forward
[YOUR CODE]               # call backward again WITHOUT zeroing
print("after 2nd backward (accumulated!):", x.grad.item())


![slide-25](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-25.jpg)

## 1.10 Disabling Gradient Tracking

Wrap updates in `torch.no_grad()` so the weight step itself is not recorded in
the graph. Forgetting this is a classic source of memory leaks in manual
training loops.

**Your task:** write a 100-iteration gradient descent loop for `f(x) = x^2`
starting from `x = 5.0` with learning rate `0.1`. Final `x` should be near 0.

In [ ]:
x = torch.tensor(5.0, requires_grad=True)
lr = 0.1

for it in range(100):
    f = [YOUR CODE]          # forward
    [YOUR CODE]              # backward

    with torch.no_grad():
        [YOUR CODE]          # gradient step: x -= lr * x.grad

    [YOUR CODE]              # zero x.grad for the next iteration

print("final x =", x.item())


---
# Part 2 — Regression + DataLoaders
California-housing regression from scratch, then with the high-level `nn` API,
then with an MLP, then with mini-batches.

![slide-33](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-33.jpg)

## 2.1 Linear Regression: Setup

Load California housing, split, standardize (SGD needs scaled features), and
convert to float tensors. The target must be a column vector, shape `(N, 1)`.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing()
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42)
X_train_np, X_valid_np, y_train_np, y_valid_np = train_test_split(
    X_train_np, y_train_np, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np)
X_valid_np = scaler.transform(X_valid_np)
X_test_np  = scaler.transform(X_test_np)

# Convert to float tensors; reshape targets to column vectors.
X_train = [YOUR CODE]    # float32 tensor from X_train_np
X_valid = [YOUR CODE]
X_test  = [YOUR CODE]

y_train = [YOUR CODE]    # hint: torch.FloatTensor(...).view(-1, 1)
y_valid = [YOUR CODE]
y_test  = [YOUR CODE]

n_features = X_train.shape[1]
print("features:", n_features, "train shape:", X_train.shape, y_train.shape)


![slide-36](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-36.jpg)

## 2.2 Low-Level Linear Regression

Manual parameters, manual gradient step. No `nn.Module`, no optimizer.

**Your task:** fill in the training loop. Update rule: `p -= lr * p.grad`
under `torch.no_grad()`, then zero each grad.

In [ ]:
torch.manual_seed(42)
w = [YOUR CODE]      # shape (n_features, 1), requires_grad=True
b = [YOUR CODE]      # scalar 0.0,              requires_grad=True

lr = 0.4
n_epochs = 20
for epoch in range(n_epochs):
    y_pred = [YOUR CODE]                         # X_train @ w + b
    loss = [YOUR CODE]                           # mean squared error
    loss.backward()

    with torch.no_grad():
        [YOUR CODE]                              # w step
        [YOUR CODE]                              # b step
        [YOUR CODE]                              # zero w.grad
        [YOUR CODE]                              # zero b.grad

    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch+1:2d}  loss={loss.item():.4f}")


![slide-37](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-37.jpg)

## 2.3 High-Level API: `nn.Linear`

Same model, much less boilerplate. `nn.Linear(in, out)` creates a fresh
`weight` (out, in) and `bias` (out,) and registers them as learnable parameters.

In [ ]:
torch.manual_seed(42)
model = [YOUR CODE]   # nn.Linear(...); same architecture as Part 2.2

# peek at the registered parameters
for name, p in model.named_parameters():
    print(name, tuple(p.shape), "requires_grad:", p.requires_grad)

# forward pass on the first two rows
print(model(X_train[:2]))


![slide-41](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-41.jpg)

## 2.4 Optimizer + Criterion

The optimizer owns the parameters; the criterion is just a loss module.

**Your task:** build an SGD optimizer on `model`'s parameters with `lr=0.1`,
and an MSE criterion.

In [ ]:
lr = 0.1
optimizer = [YOUR CODE]   # hint: torch.optim.SGD
criterion = [YOUR CODE]   # hint: nn.MSELoss()


![slide-42](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-42.jpg)

## 2.5 High-Level Training Loop

The canonical 4-line step: `forward -> loss -> backward -> step (+ zero)`.

**Your task:** implement `train_bgd`.

In [ ]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = [YOUR CODE]
        loss   = [YOUR CODE]
        [YOUR CODE]            # backward
        [YOUR CODE]            # step
        [YOUR CODE]            # zero grads

        if (epoch + 1) % 5 == 0:
            print(f"epoch {epoch+1:2d}  loss={loss.item():.4f}")

train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs=20)


![slide-44](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-44.jpg)

## 2.6 MLP via `nn.Sequential`

Linear regression can only fit hyperplanes. Stack linears + nonlinearities to
get more capacity.

**Your task:** build a 3-hidden-layer MLP `n -> 50 -> 40 -> 30 -> 1` with ReLU
between each pair of linears (no activation after the last linear).

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(
    [YOUR CODE],   # Linear n_features -> 50
    nn.ReLU(),
    [YOUR CODE],   # Linear 50 -> 40
    nn.ReLU(),
    [YOUR CODE],   # Linear 40 -> 30
    nn.ReLU(),
    [YOUR CODE],   # Linear 30 -> 1  (no ReLU after)
)
print(model)


![slide-46](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-46.jpg)

## 2.7 Mini-Batch GD: Dataset + DataLoader

`TensorDataset` wraps aligned tensors into a `(len, getitem)` dataset.
`DataLoader` handles batching and shuffling for you.

**Your task:** wrap `(X_train, y_train)` into a TensorDataset, then build a
DataLoader with `batch_size=32`, `shuffle=True`.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = [YOUR CODE]   # hint: TensorDataset(...)
train_loader  = [YOUR CODE]   # hint: DataLoader(dataset, batch_size=..., shuffle=...)

valid_dataset = TensorDataset(X_valid, y_valid)
valid_loader  = DataLoader(valid_dataset, batch_size=32)

# peek: number of batches, shape of first batch
print("n batches:", len(train_loader))
xb, yb = next(iter(train_loader))
print("batch shapes:", xb.shape, yb.shape)


![slide-50](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-50.jpg)

## 2.8 Mini-Batch Training Loop (GPU-Ready)

Two orthogonal switches: `model.train()` (layer mode) and `.to(device)`
(data location). Move the model to device once; move each batch inside the
loop.

**Your task:** implement `train`. Print the epoch mean loss each epoch.

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50), nn.ReLU(),
    nn.Linear(50, 40), nn.ReLU(),
    nn.Linear(40, 1),
).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.02)
criterion = nn.MSELoss()


def train(model, optimizer, criterion, train_loader, n_epochs):
    [YOUR CODE]                          # put model in training mode
    for epoch in range(n_epochs):
        total = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = [YOUR CODE]   # move both to device
            y_pred = [YOUR CODE]
            loss   = [YOUR CODE]
            total += loss.item()
            [YOUR CODE]                      # backward
            [YOUR CODE]                      # optimizer step
            [YOUR CODE]                      # zero grads
        print(f"epoch {epoch+1:2d}  avg loss = {total/len(train_loader):.4f}")


train(model, optimizer, criterion, train_loader, n_epochs=5)


---
# Part 3 — Evaluation
Evaluation is not training. Two switches (`model.eval()`, `torch.no_grad()`) and
the right aggregation.

![slide-56](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-56.jpg)

## 3.1 An Evaluation Function

Same shape as training but **no** `loss.backward()`, **no** `optimizer.step()`,
**no** `zero_grad()`.

**Your task:** implement `evaluate`. Collect one metric scalar per batch, stack,
then aggregate.

In [ ]:
def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    [YOUR CODE]                           # put model in eval mode
    metrics = []
    with [YOUR CODE]:                     # turn gradient tracking off
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            m = metric_fn(y_pred, y_batch)
            metrics.append(m)
    return aggregate_fn(torch.stack(metrics))


mse_fn = nn.MSELoss()
valid_mse = evaluate(model, valid_loader, mse_fn)
print("valid MSE:", valid_mse.item())


![slide-58](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-58.jpg)

## 3.2 Mean of Batch RMSEs != Overall RMSE

`mean(sqrt(x)) <= sqrt(mean(x))` (Jensen). Averaging per-batch RMSEs is biased.

**Your task:** pass a custom `aggregate_fn` that takes MSE values per batch and
returns the square root of their mean — the correct per-population RMSE.

In [ ]:
valid_rmse = evaluate(model, valid_loader, mse_fn,
                      aggregate_fn=lambda ms: [YOUR CODE])  # sqrt of the mean

print("valid RMSE (correct):", valid_rmse.item())


![slide-59](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-59.jpg)

## 3.3 `torchmetrics`: update + compute

The library separates linear accumulation (`update`) from the nonlinear finish
(`compute`). You call `reset()` between evaluations.

**Your task:** create a `torchmetrics.MeanSquaredError(squared=False)` metric
and wire it into `evaluate_tm`.

In [ ]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    [YOUR CODE]                       # reset the metric
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            [YOUR CODE]               # feed this batch into the metric
    return [YOUR CODE]                # final scalar


rmse = [YOUR CODE].to(device)         # torchmetrics.MeanSquaredError(...)
print("valid RMSE (torchmetrics):", evaluate_tm(model, valid_loader, rmse).item())


---
# Part 4 — Custom Modules
Subclass `nn.Module` when `Sequential` isn't enough: skip connections, Wide &
Deep, multi-input, multi-output.

![slide-62](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-62.jpg)

## 4.1 Subclassing `nn.Module`

Register submodules in `__init__` (assignments auto-register); describe the
computation in `forward`.

**Your task:** implement a two-hidden-layer MLP as a subclass.

In [ ]:
class MyModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.layer1 = [YOUR CODE]    # Linear n_features -> 50
        self.layer2 = [YOUR CODE]    # Linear 50 -> 1

    def forward(self, X):
        h = [YOUR CODE]              # relu(layer1(X))
        return [YOUR CODE]           # layer2(h)


torch.manual_seed(42)
m = MyModel(n_features)
print(m)
print("output on first row:", m(X_train[:1]))


![slide-63](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-63.jpg)

## 4.2 The Wide & Deep Architecture

Wide path: linear skip of raw features (memorization). Deep path: MLP
(generalization). Concatenate both, apply one final linear layer.

![slide-64](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-64.jpg)

## 4.2 (cont.) Implementing Wide & Deep

**Your task:** implement `WideAndDeep`. The deep stack is `n -> 50 -> 40 -> 30`
with ReLU between each pair; the output layer takes `[X, deep(X)]` (shape
`n + 30`) to 1 output.

In [ ]:
class WideAndDeep(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            [YOUR CODE], nn.ReLU(),
            [YOUR CODE], nn.ReLU(),
            [YOUR CODE], nn.ReLU(),
        )
        self.output_layer = [YOUR CODE]   # Linear (n_features + 30) -> 1

    def forward(self, X):
        d = [YOUR CODE]                   # run the deep stack
        concat = [YOUR CODE]              # torch.concat([X, d], dim=1)
        return [YOUR CODE]                # self.output_layer(concat)


torch.manual_seed(42)
wd = WideAndDeep(n_features)
print(wd)
print("output shape on first 3 rows:", wd(X_train[:3]).shape)


![slide-65](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-65.jpg)

## 4.3 Container Modules

When the number of submodules is variable, use `nn.ModuleList` or
`nn.ModuleDict` instead of a plain Python list/dict — otherwise PyTorch will
NOT register them as parameters.

**Your task:** build a flexible MLP whose hidden widths come from a list.

In [ ]:
class FlexibleMLP(nn.Module):
    def __init__(self, n_in, hiddens, n_out):
        super().__init__()
        dims = [n_in] + list(hiddens) + [n_out]
        self.layers = [YOUR CODE]    # nn.ModuleList of nn.Linear(dims[i], dims[i+1])
        self.act = nn.ReLU()

    def forward(self, X):
        for i, layer in enumerate(self.layers):
            X = layer(X)
            if i < len(self.layers) - 1:   # no activation after last layer
                X = self.act(X)
        return X


m = FlexibleMLP(n_features, [64, 32, 16], 1)
print(m)
print(sum(p.numel() for p in m.parameters()), "total parameters")


![slide-66](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-66.jpg)

## 4.4 Multiple Inputs

Give `forward` more than one positional argument. The DataLoader must yield
tuples matching that signature.

**Your task:** implement `WideAndDeepV3` where 5 columns feed the wide path
and the remaining `n - 2` columns feed the deep path.

In [ ]:
class WideAndDeepV3(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features - 2, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
            nn.Linear(40, 30), nn.ReLU(),
        )
        self.output_layer = [YOUR CODE]    # Linear (30 + 5) -> 1

    def forward(self, X_wide, X_deep):
        d = [YOUR CODE]
        concat = [YOUR CODE]               # torch.concat([X_wide, d], dim=1)
        return [YOUR CODE]


torch.manual_seed(42)
wd3 = WideAndDeepV3(n_features)
# two input batches of matching widths:
Xw = torch.randn(4, 5)
Xd = torch.randn(4, n_features - 2)
print(wd3(Xw, Xd).shape)


![slide-68](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-68.jpg)

## 4.5 Multiple Outputs + Auxiliary Loss

A second head off the deep tower acts as a regularizer: the deep stack must
produce features that are simultaneously useful alone (aux) and with the wide
path (main). Train with a weighted combination of the two losses.

**Your task:** implement `WideAndDeepV4` with a main output `Linear(30 + 5, 1)`
and an auxiliary output `Linear(30, 1)`.

In [ ]:
class WideAndDeepV4(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features - 2, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
            nn.Linear(40, 30), nn.ReLU(),
        )
        self.output_layer = [YOUR CODE]     # main head
        self.aux_output_layer = [YOUR CODE] # aux head off the deep stack

    def forward(self, X_wide, X_deep):
        d = self.deep_stack(X_deep)
        main = [YOUR CODE]
        aux  = [YOUR CODE]
        return main, aux


torch.manual_seed(42)
wd4 = WideAndDeepV4(n_features)
Xw = torch.randn(4, 5); Xd = torch.randn(4, n_features - 2)
main, aux = wd4(Xw, Xd)
print("main:", main.shape, "  aux:", aux.shape)


---
# Part 5 — Image Classification
FashionMNIST, a flatten-based MLP classifier, logits + CrossEntropyLoss,
top-k inference.

![slide-71](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-71.jpg)

## 5.1 TorchVision + Loading FashionMNIST

`transforms.v2.ToImage()` turns a PIL/array into a torchvision Image tensor;
`ToDtype(float32, scale=True)` casts to float and divides by 255.

**Your task:** build the transform pipeline, then download train and test
datasets.

In [ ]:
import torchvision
import torchvision.transforms.v2 as T

toTensor = T.Compose([
    [YOUR CODE],     # T.ToImage()
    [YOUR CODE],     # T.ToDtype(torch.float32, scale=True)
])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True,  download=True, transform=toTensor)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=toTensor)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000])

print("train/valid/test sizes:",
      len(train_data), len(valid_data), len(test_data))


![slide-72](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-72.jpg)

## 5.2 Channel-First Layout

PyTorch images are `[C, H, W]` (not `[H, W, C]` like matplotlib/OpenCV).
FashionMNIST is grayscale -> `C = 1`.

**Your task:** grab the first sample, inspect its shape/dtype, then display it
with matplotlib using `permute` to convert to HWC.

In [ ]:
X_sample, y_sample = train_data[0]

print("shape:", X_sample.shape)         # expect [1, 28, 28]
print("dtype:", X_sample.dtype)         # expect float32 in [0, 1]
print("label:", train_and_valid_data.classes[y_sample])

# matplotlib wants HWC. hint: X_sample.permute(...)
plt.imshow([YOUR CODE], cmap="gray")
plt.axis("off")
plt.title(train_and_valid_data.classes[y_sample])
plt.show()


![slide-74](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-74.jpg)

## 5.3 Building the Classifier

MLP classifier over flattened pixels: `[B, 1, 28, 28] -> [B, 784] -> [B, H1]
-> [B, H2] -> [B, 10]`.

**Your task:** build `ImageClassifier` with `nn.Flatten()` + MLP head.

In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            [YOUR CODE],                 # nn.Flatten()
            [YOUR CODE],                 # Linear(n_inputs, n_hidden1)
            nn.ReLU(),
            [YOUR CODE],                 # Linear(n_hidden1, n_hidden2)
            nn.ReLU(),
            [YOUR CODE],                 # Linear(n_hidden2, n_classes)  -- no activation
        )

    def forward(self, X):
        return self.mlp(X)


torch.manual_seed(42)
model = ImageClassifier(n_inputs=1 * 28 * 28,
                        n_hidden1=300, n_hidden2=100,
                        n_classes=10).to(device)
print(model)


![slide-75](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-75.jpg)

## 5.4 Why no softmax on the output?

`nn.CrossEntropyLoss` already fuses `log_softmax + NLL` internally for numerical
stability (log-sum-exp). Double-softmax is a silent bug. Keep the model's
output as raw logits.

**Your task (no code):** in one sentence, why does temperature scaling also
require raw logits?

![slide-76](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-76.jpg)

## 5.5 `nn.Flatten`: Preparing Inputs for Linear Layers

`nn.Flatten()` collapses all dims after batch into one: `[B, 1, 28, 28] -> [B, 784]`.
Zero parameters.

**Your task:** manually verify the shape transform.

In [ ]:
xb = torch.zeros(16, 1, 28, 28)
flat = nn.Flatten()
out = [YOUR CODE]            # run flat on xb
print("before:", xb.shape, "  after:", out.shape)


![slide-77](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-77.jpg)

## 5.6 Classification Loss Cheat Sheet

| Task | Output | Loss |
|---|---|---|
| Binary classification | 1 logit | `BCEWithLogitsLoss` |
| Multi-class (one label) | C logits | `CrossEntropyLoss` |
| Multi-label | C logits | `BCEWithLogitsLoss` |
| Regression | k values | `MSELoss` / `L1Loss` |

Pattern: use `*WithLogits*` losses so the model's forward keeps raw logits.

![slide-78](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-78.jpg)

## 5.7 Making Predictions (Top-1 and Top-k)

At inference time: `model.eval()`, `no_grad()`, then `argmax(dim=1)` for Top-1
or `torch.topk(logits, k=K, dim=1)` for the top K classes.

**Your task:** predict on 3 validation images.

In [ ]:
# (This cell expects `model` to have been trained — skip the `values` assertion
# if you haven't trained yet.)
model.eval()
from torch.utils.data import DataLoader
valid_loader = DataLoader(valid_data, batch_size=32)

X_new, y_new = next(iter(valid_loader))
X_new, y_new = X_new[:3].to(device), y_new[:3]

with torch.no_grad():
    logits = [YOUR CODE]           # model(X_new)

y_pred_top1 = [YOUR CODE]          # logits.argmax(dim=1)
top4_vals, top4_idx = [YOUR CODE]  # torch.topk(logits, k=4, dim=1)

print("true:", y_new.tolist())
print("top-1:", y_pred_top1.cpu().tolist())
print("top-4 indices:\n", top4_idx.cpu())


---
# Part 6 — Production Readiness
Hyperparameter tuning (Optuna), saving/loading, security, TorchScript, and
`torch.compile`.

![slide-81](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-81.jpg)

## 6.1 Optuna: The Objective Function

An objective takes a `trial`, samples hyperparameters, trains, and returns a
scalar to be maximized (or minimized).

**Your task:** sketch an objective that searches `learning_rate` in
`[1e-5, 1e-1]` on a log scale and `n_hidden` as an int in `[20, 300]`.

In [ ]:
import optuna

def objective(trial):
    learning_rate = trial.suggest_float([YOUR CODE])   # "learning_rate", low, high, log=True
    n_hidden      = trial.suggest_int  ([YOUR CODE])   # "n_hidden",      low, high

    model = ImageClassifier(n_inputs=1*28*28,
                            n_hidden1=n_hidden, n_hidden2=n_hidden,
                            n_classes=10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    # ... train for a few epochs, return best validation accuracy.
    # (Leave the inner training out in this sketch; return a dummy value.)
    return float(learning_rate * n_hidden)   # placeholder


![slide-82](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-82.jpg)

## 6.2 Running an Optuna Study

`TPESampler` with a fixed seed so trials are reproducible. 5 trials is a quick
smoke test; real studies use 50-200.

**Your task:** create the study (maximize) and run 5 trials.

In [ ]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
study = [YOUR CODE]         # optuna.create_study(direction=..., sampler=...)
[YOUR CODE]                 # study.optimize(objective, n_trials=5)

print("best params:", study.best_params)
print("best value :", study.best_value)


![slide-84](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-84.jpg)

## 6.3 Why pickle is risky (and what to do)

`torch.save` / `torch.load` go through `pickle`. Unpickling an untrusted file
runs arbitrary code. Two defenses:

1. Prefer saving just `state_dict()` (tensor weights, not arbitrary Python).
2. When loading, pass `weights_only=True` so only tensor deserialization is
   allowed.

In [ ]:
# Round-trip the full model (risky pattern): save then load.
torch.save(model, "my_fashion_mnist.pt")

# Show both load modes.
# 1. SAFE-ish: weights_only=True — but this fails for full-model files.
try:
    [YOUR CODE]                    # torch.load(..., weights_only=True)
except Exception as e:
    print("expected failure with weights_only=True:", type(e).__name__)

# 2. Full-model load: only use this on files you created yourself.
loaded_full = torch.load("my_fashion_mnist.pt", weights_only=False)
print(type(loaded_full).__name__)


![slide-85](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-85.jpg)

## 6.4 Recommended: `state_dict` only

Save tensors; reconstruct the architecture in code on load. Safe and portable.

**Your task:** save `state_dict`, rebuild `ImageClassifier`, load weights,
verify predictions match the original.

In [ ]:
# Save weights only.
[YOUR CODE]                        # torch.save(model.state_dict(), "weights.pt")

# Rebuild + load.
new_model = ImageClassifier(n_inputs=1*28*28, n_hidden1=300, n_hidden2=100,
                            n_classes=10)
loaded_weights = [YOUR CODE]       # torch.load("weights.pt", weights_only=True)
[YOUR CODE]                        # new_model.load_state_dict(...)
new_model.eval()

with torch.no_grad():
    same = torch.allclose(model(X_new).cpu(), new_model(X_new.cpu()))
print("outputs match:", same)


![slide-86](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-86.jpg)

## 6.5 Save Architecture Alongside Weights

Bundle the construction hyperparameters with the state dict so reloads don't
need source code that hardcodes sizes.

In [ ]:
model_data = {
    "model_state_dict": [YOUR CODE],     # model.state_dict()
    "model_hyperparameters": {
        "n_inputs":   1 * 28 * 28,
        "n_hidden1":  300,
        "n_hidden2":  100,
        "n_classes":  10,
    },
}
torch.save(model_data, "my_fashion_mnist_model.pt")

loaded = [YOUR CODE]                     # torch.load(..., weights_only=True)
rebuilt = ImageClassifier(**loaded["model_hyperparameters"])
rebuilt.load_state_dict(loaded["model_state_dict"])
rebuilt.eval()
print(rebuilt)


![slide-87](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-87.jpg)

## 6.6 Checkpointing: Resumable Training

A checkpoint is `state_dict`s for model + optimizer plus some metadata
(`epoch`, `loss history`, RNG state if you want strict reproducibility).

In [ ]:
ckpt = {
    "epoch": 3,
    "model_state_dict":     [YOUR CODE],   # model.state_dict()
    "optimizer_state_dict": [YOUR CODE],   # optimizer.state_dict()
    "loss_history": [0.72, 0.41, 0.28],
}
torch.save(ckpt, "ckpt.pt")

# Later: resume.
resumed = torch.load("ckpt.pt", weights_only=True)
print("resumed at epoch", resumed["epoch"])


## 6.7 TorchScript: trace vs script

Convert a Python `nn.Module` into a serialized IR so it can run outside Python
(C++ / LibTorch, mobile, production servers).

- `torch.jit.trace(m, example_input)` — records one concrete forward pass.
  Control flow that depends on data will be frozen at the traced value.
- `torch.jit.script(m)` — reads your Python source, preserves control flow.

**Your task:** export the model both ways, optimize for inference, save, and
reload.

In [ ]:
# We need a concrete input for trace(). Use X_new from earlier.
traced  = [YOUR CODE]                   # torch.jit.trace(model, X_new)
scripted = [YOUR CODE]                  # torch.jit.script(model)

optimized = [YOUR CODE]                 # torch.jit.optimize_for_inference(scripted)
[YOUR CODE]                             # optimized.save("my_fashion_mnist_ts.pt")

reloaded = [YOUR CODE]                  # torch.jit.load("my_fashion_mnist_ts.pt")

with torch.no_grad():
    a = model(X_new).cpu()
    b = reloaded(X_new).cpu()
print("TorchScript matches eager:", torch.allclose(a, b, atol=1e-5))


## 6.8 `torch.compile`

Same API, but the first call captures a graph and fuses kernels (Dynamo ->
AOTAutograd -> Inductor -> Triton/C++). Subsequent calls run the compiled
kernel.

**Your task:** wrap `model` with `torch.compile`, then run it once on
`X_new` to trigger compilation. (On CPU the win is modest; on GPU you
should see a clear speedup after warmup.)

In [ ]:
compiled_model = [YOUR CODE]           # torch.compile(model)

if device == "cuda":
    # warmup
    with torch.no_grad():
        _ = compiled_model(X_new)
        torch.cuda.synchronize()

    # timed run
    import time
    t0 = time.time()
    with torch.no_grad():
        for _ in range(100):
            _ = compiled_model(X_new)
        torch.cuda.synchronize()
    print(f"100 compiled forwards: {time.time()-t0:.3f}s")
else:
    with torch.no_grad():
        _ = compiled_model(X_new)
    print("ran one compiled forward on CPU")


![slide-92](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/slides/slide-92.jpg)

# Done

**PyTorch in one slide (from the deck):**
- Tensors = NumPy + GPU + autograd.
- `nn.Module` = LEGO bricks; `nn.Sequential` for straight stacks, subclass for
  everything else (skips, multi-input, multi-output).
- Training is 4 lines: forward, loss, backward, step (+ zero).
- Eval is NOT training: `model.eval()` + `torch.no_grad()`.
- `torchmetrics` for non-linear metrics.
- Save `state_dict` (safe); avoid full-model pickle from untrusted sources;
  prefer `weights_only=True` and `safetensors`.
- `torch.jit.{trace, script}` to leave Python; `torch.compile` to fuse kernels.
